### Split do artigo vs. dataset completo

O artigo usa só 1.102 das 2.208 amostras de `SANEPAR_COMPLETO.txt` (601 estimação + 501 validação, `notebooks/processing/01_mimo_dataset_from_papper.ipynb`). Aqui comparamos esse recorte com o dataset completo e medimos o modelo ARMAX/MOGWO reproduzido (`notebooks/modeling/01_mogwo_armax_reproducao.ipynb`) fora da janela em que foi ajustado.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DATA_RAW = Path("../../data/raw")
DATA_EXTERNAL = Path("../../data/external")
DATA_PROCESSED = Path("../../data/processed")

SANEPAR_PATH = (
    DATA_RAW
    / "WRSO and NSWRSO_draft_all"
    / "Multi_objective_for_MIMO_systems"
    / "MOGWO"
    / "SANEPAR_COMPLETO.txt"
)
TABELA3_PATH = DATA_EXTERNAL / "tabela3_parametros_armax_papper.csv"

COLUMNS = [
    "freq_b1",
    "freq_b2",
    "freq_b3",
    "vazao_entrada",
    "vazao_distribuicao",
    "nivel_res",
    "pressao",
]
INPUT_COLS = ["freq_b1", "freq_b2", "freq_b3", "vazao_entrada"]
OUTPUT_COLS = ["vazao_distribuicao", "nivel_res", "pressao"]

TRAIN_SLICE = slice(549, 1150)  # est = 550:1150 (MATLAB 1-indexado)
TEST_SLICE = slice(1499, 2000)  # val = 1500:2000

### Carregar dataset completo e recorte do artigo

`SANEPAR_COMPLETO.txt` tem 2.208 amostras no total; o artigo usa só as linhas `550:1150` (treino) e `1500:2000` (teste), 1.102 no total — o restante (~50%) nunca é visto pelo modelo publicado.

In [2]:
full_df = pd.DataFrame(
    np.loadtxt(SANEPAR_PATH), columns=COLUMNS
)
full_df.index.name = "sample_idx"

paper_idx = np.concatenate(
    [np.arange(*TRAIN_SLICE.indices(len(full_df))),
     np.arange(*TEST_SLICE.indices(len(full_df)))]
)
paper_df = full_df.loc[paper_idx]
outside_df = full_df.drop(index=paper_idx)

print("completo:", full_df.shape)
print("usado no artigo (treino+teste):", paper_df.shape)
print("fora do recorte do artigo:", outside_df.shape)
print(
    f"fração usada: {len(paper_df) / len(full_df):.1%}"
)

completo: (2208, 7)
usado no artigo (treino+teste): (1102, 7)
fora do recorte do artigo: (1106, 7)
fração usada: 49.9%


### Estatísticas descritivas: recorte do artigo vs. completo

In [3]:
summary = pd.concat(
    {
        "completo": full_df[OUTPUT_COLS + INPUT_COLS].describe(),
        "recorte_artigo": paper_df[OUTPUT_COLS + INPUT_COLS].describe(),
        "fora_recorte": outside_df[OUTPUT_COLS + INPUT_COLS].describe(),
    },
    axis=1,
)
summary

completo                                                      \
      vazao_distribuicao    nivel_res      pressao      freq_b1      freq_b2   
count        2208.000000  2208.000000  2208.000000  2208.000000  2208.000000   
mean           67.644960     2.403007    22.173296    31.265935    35.868844   
std            25.764200     0.295912     4.249852    25.198973    23.849093   
min           -20.107826     0.007118     0.231240     0.000000     0.000000   
25%            41.560803     2.252698    19.636037     0.000000     5.000000   
50%            70.224648     2.442053    21.920000    46.221830    48.998888   
75%            88.786425     2.603691    25.989245    53.799999    54.520961   
max           199.590027     3.029920    30.290001    60.000000    60.000000   

                                     recorte_artigo                            \
           freq_b3 vazao_entrada vazao_distribuicao    nivel_res      pressao   
count  2208.000000   2208.000000        1102.000000  1102.000000  1102.000000   
mean      3.392681     66.157330          68.521538     2.399430    22.348356   
std      11.912819     34.236532          25.794528     0.249043     4.106434   
min       0.000000     -0.000050         -20.107826     1.603524     0.650000   
25%       0.000000     37.749400          43.394124     2.231538    19.720001   
50%       0.000000     68.427586          71.230789     2.431610    21.980001   
75%       0.000000     95.855494          89.970140     2.573336    26.040001   
max      60.000000    200.162064         119.193413     3.011450    28.200001   

       ...                                               fora_recorte  \
       ...      freq_b2      freq_b3 vazao_entrada vazao_distribuicao   
count  ...  1102.000000  1102.000000   1102.000000        1106.000000   
mean   ...    31.475979     1.166885     68.126078          66.771551   
std    ...    25.343199     6.554061     33.186692          25.715888   
min    ...     0.000000     0.000000     17.082411           0.268883   
25%    ...     0.000000     0.000000     38.846274          40.666569   
50%    ...    46.799999     0.000000     69.338043          69.206493   
75%    ...    53.899998     0.000000     95.887356          87.430902   
max    ...    60.000000    57.299999    150.262070         199.590027   

                                                                        \
         nivel_res      pressao      freq_b1      freq_b2      freq_b3   
count  1106.000000  1106.000000  1106.000000  1106.000000  1106.000000   
mean      2.406570    21.998869    23.740447    40.245822     5.610427   
std       0.336254     4.382999    25.531411    21.392554    15.191493   
min       0.007118     0.231240     0.000000     0.000000     0.000000   
25%       2.263726    19.562617     0.000000    29.807901     0.000000   
50%       2.455861    21.730229     6.019001    50.321508     0.000000   
75%       2.629509    25.930000    51.599278    55.258607     0.000000   
max       3.029920    30.290001    60.000000    60.000000    60.000000   

                     
      vazao_entrada  
count   1106.000000  
mean      64.195702  
std       35.156953  
min       -0.000050  
25%       32.600085  
50%       66.385563  
75%       95.769777  
max      200.162064  

[8 rows x 21 columns]

### Séries completas, com o recorte do artigo destacado

Áreas sombreadas marcam `treino` (550:1150) e `teste` (1500:2000) do artigo; o resto da série nunca entra no ajuste nem na validação publicados.

In [4]:
labels = {
    "vazao_distribuicao": "Vazão de distribuição (l/s)",
    "nivel_res": "Nível (m)",
    "pressao": "Pressão (mca)",
}

fig = make_subplots(rows=3, cols=1, subplot_titles=list(labels.values()))
for i, (col, label) in enumerate(labels.items()):
    fig.add_trace(
        go.Scatter(
            x=full_df.index,
            y=full_df[col],
            name="Completo",
            line=dict(color="black", width=1),
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    for name, sl, color in (
        ("Treino (artigo)", TRAIN_SLICE, "rgba(31,119,180,0.25)"),
        ("Teste (artigo)", TEST_SLICE, "rgba(214,39,40,0.25)"),
    ):
        fig.add_vrect(
            x0=sl.start,
            x1=sl.stop,
            fillcolor=color,
            line_width=0,
            row=i + 1,
            col=1,
        )
fig.update_layout(
    title_text="Dataset completo (2208 amostras) — recorte do artigo em destaque",
    height=800,
    template="plotly_white",
)
fig.show()

### Distribuição: recorte do artigo vs. restante da série

Se o recorte do artigo não for representativo do regime operacional completo, o modelo ajustado nele deve degradar fora dessa janela — testado na seção seguinte.

In [5]:
fig = make_subplots(rows=1, cols=3, subplot_titles=list(labels.values()))
for i, (col, label) in enumerate(labels.items()):
    fig.add_trace(
        go.Histogram(
            x=paper_df[col],
            name="Recorte artigo",
            marker_color="tab:blue" if False else "royalblue",
            opacity=0.6,
            showlegend=(i == 0),
        ),
        row=1,
        col=i + 1,
    )
    fig.add_trace(
        go.Histogram(
            x=outside_df[col],
            name="Fora do recorte",
            marker_color="firebrick",
            opacity=0.6,
            showlegend=(i == 0),
        ),
        row=1,
        col=i + 1,
    )
fig.update_layout(
    barmode="overlay",
    title_text="Distribuição por saída: recorte do artigo vs. resto",
    height=400,
    template="plotly_white",
)
fig.show()

---
### Avaliar o ARMAX/MOGWO (parâmetros da Tabela 3) no dataset completo

Reaplica a mesma `armax_predict`/`armax_objective` de `notebooks/modeling/01_mogwo_armax_reproducao.ipynb` (idêntica, verificada contra o `.mat` do MATLAB) sobre as 2.208 amostras completas, para medir o modelo publicado fora da janela em que foi ajustado/validado.

In [6]:
N_PARAMS = 72
N_LAGS = 3
N_OUTPUTS = 3


def _lag(x: np.ndarray, lag: int) -> np.ndarray:
    """Lag a (tam,) or (tam, pop) array by `lag` steps, zero-filled."""
    shifted = np.zeros_like(x)
    shifted[lag:] = x[:-lag]
    return shifted


def armax_predict(x: np.ndarray, u: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Predição one-step ARMAX (equação 15), idêntica à da reprodução
    em `notebooks/modeling/01_mogwo_armax_reproducao.ipynb`."""
    tam = y.shape[1]
    pop = x.shape[1]

    y_lags = [[_lag(y[i], lag) for lag in (1, 2, 3)] for i in range(3)]
    u_lags = [[_lag(u[i], lag) for lag in (1, 2, 3)] for i in range(4)]

    y_pred = np.zeros((3, tam, pop))
    for out in range(3):
        base = out * 24
        a_self = x[base : base + 3]
        others = [o for o in range(3) if o != out]
        a_other1 = x[base + 3 : base + 6]
        a_other2 = x[base + 6 : base + 9]
        b_u = x[base + 9 : base + 21].reshape(4, 3, pop)
        c_e = x[base + 21 : base + 24]

        pred = np.zeros((tam, pop))
        for lag_idx, lag in enumerate((1, 2, 3)):
            pred -= np.outer(y_lags[out][lag_idx], a_self[lag_idx])
            pred -= np.outer(y_lags[others[0]][lag_idx], a_other1[lag_idx])
            pred -= np.outer(y_lags[others[1]][lag_idx], a_other2[lag_idx])
            for inp in range(4):
                pred += np.outer(u_lags[inp][lag_idx], b_u[inp, lag_idx])
        pred[:3] = y[out, :3, None]

        e = y[out][:, None] - pred
        e_lags = [_lag(e, lag) for lag in (1, 2, 3)]
        correction = sum(e_lags[k] * c_e[k] for k in range(3))

        y_pred[out] = pred + correction

    return y_pred


def metrics_per_output(
    x: np.ndarray, u: np.ndarray, y: np.ndarray
) -> pd.DataFrame:
    """MSE, MAE, R² e SMAPE por saída, para uma única solução (pop=1)."""
    y_pred = armax_predict(x, u, y)[:, :, 0]  # (3, tam)

    error = y - y_pred
    mse = np.mean(error**2, axis=1)
    mae = np.mean(np.abs(error), axis=1)

    y_mean = y.mean(axis=1, keepdims=True)
    ss_res = np.sum(error**2, axis=1)
    ss_tot = np.sum((y - y_mean) ** 2, axis=1)
    r2 = 1 - ss_res / ss_tot

    smape = 100 * np.mean(
        2
        * np.abs(error)
        / (np.abs(y) + np.abs(y_pred) + np.finfo(float).eps),
        axis=1,
    )

    return pd.DataFrame(
        {"mse": mse, "mae": mae, "r2": r2, "smape_%": smape},
        index=OUTPUT_COLS,
    )


tabela3 = pd.read_csv(TABELA3_PATH)
x_mogwo = tabela3["MOGWO"].to_numpy().reshape(N_PARAMS, 1)

In [7]:
U_full = full_df[INPUT_COLS].to_numpy().T
Y_full = full_df[OUTPUT_COLS].to_numpy().T
U_paper = paper_df[INPUT_COLS].to_numpy().T
Y_paper = paper_df[OUTPUT_COLS].to_numpy().T

metrics_paper = metrics_per_output(x_mogwo, U_paper, Y_paper)
metrics_full = metrics_per_output(x_mogwo, U_full, Y_full)

comparison = pd.concat(
    {"recorte_artigo": metrics_paper, "dataset_completo": metrics_full},
    axis=1,
)
comparison.round(4)

recorte_artigo                         dataset_completo  \
                              mse     mae      r2 smape_%              mse   
vazao_distribuicao        41.1491  4.4502  0.9381  7.8995          80.0557   
nivel_res                  0.0029  0.0355  0.9540  1.5420           0.0120   
pressao                    2.5634  1.0043  0.8478  5.0425           3.6138   

                                            
                       mae      r2 smape_%  
vazao_distribuicao  4.7907  0.8793  8.5243  
nivel_res           0.0468  0.8624  2.3398  
pressao             1.1103  0.7998  5.8773

O recorte do artigo reaplica o próprio treino+teste publicado (serve de sanity check — deve bater com a Tabela 2 do artigo). `mse/r2_dataset_completo` mostra como o mesmo modelo se comporta nas 2.208 amostras, incluindo o regime nunca visto no ajuste.

In [8]:
fig = make_subplots(rows=3, cols=1, subplot_titles=list(labels.values()))
y_pred_full = armax_predict(x_mogwo, U_full, Y_full)[:, :, 0]
for i, (col, label) in enumerate(labels.items()):
    fig.add_trace(
        go.Scatter(
            y=Y_full[i],
            name="Medido",
            line=dict(color="black", width=1),
            legendgroup="medido",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            y=y_pred_full[i],
            name="ARMAX/MOGWO (Tabela 3)",
            line=dict(color="firebrick", width=1),
            legendgroup="predito",
            showlegend=(i == 0),
        ),
        row=i + 1,
        col=1,
    )
    for name, sl, color in (
        (None, TRAIN_SLICE, "rgba(31,119,180,0.15)"),
        (None, TEST_SLICE, "rgba(214,39,40,0.15)"),
    ):
        fig.add_vrect(
            x0=sl.start, x1=sl.stop, fillcolor=color, line_width=0,
            row=i + 1, col=1,
        )
    fig.update_yaxes(title_text=label, row=i + 1, col=1)
fig.update_layout(
    title_text=(
        "ARMAX/MOGWO (Tabela 3) no dataset completo — "
        "áreas = janela de treino/teste do artigo"
    ),
    height=800,
    template="plotly_white",
    hovermode="x unified",
)
fig.show()